<a href="https://colab.research.google.com/github/A01684174/Advanced-Machine-Learning-Methods/blob/main/Equipo55_chatbot_LLM_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof. Luis Eduardo Falcón Morales

### **Actividad en Equipos: sistema LLM + RAG con información tabular**



* **Nombres y matrículas:**

*   Juan Carlos Gaibor Valencia - A01684174
*   Andrea Carolina Flores Ramirez - A01350993
*   Andrea Kimberly Fray Tamayo - A01797168
*   Pablo Gabriel Galeana Benitez - A01735281

* **Número de Equipo:** 55

### **Introducción**

En esta actividad se implementa un chatbot basado en **Retrieval-Augmented Generation (RAG)**
capaz de responder preguntas técnicas usando como base de conocimiento dos PDF:
`python_cheatsheet.pdf` y `ml_cheatsheet.pdf`.

El reto central no es el LLM, sino la **extracción y recuperación de información tabular**.
Gran parte del contenido —sobre todo el de Machine Learning— está organizado en una matriz de
*Algoritmo × {Descripción, Casos de uso, Ventajas, Desventajas}*. Un extractor de texto lineal
destruye esa estructura y provoca alucinaciones o respuestas vacías, por lo que la solución se
centra en preservar el *layout* de las tablas.

**Arquitectura de la solución:**

1. **Extracción consciente del layout.** Texto general con **PyMuPDF**; las tablas de la hoja de
   ML se detectan con **pdfplumber** y se **convierten a Markdown**, de modo que cada algoritmo
   conserve juntas su descripción, casos de uso, ventajas y desventajas.
2. **Chunking que respeta los límites de tabla.** No se parte una fila de algoritmo a la mitad:
   cada tabla Markdown es un chunk atómico con sus metadatos. Adicionalmente, cada celda de la
   tabla (Description / Use Cases / Advantages / Disadvantages) se indexa como sub-chunk para
   permitir recuperación dirigida al campo.
3. **Embeddings densos** con `all-MiniLM-L6-v2` (búsqueda semántica) indexados en **FAISS**.
4. **LLM generativo** `flan-t5-base`, con un prompt que lo obliga a responder solo desde el
   contexto recuperado, y **recuperación dirigida al campo** (*field-aware*) para preguntas por
   ventajas, desventajas o casos de uso de un algoritmo.
5. **Extracción determinista** para preguntas que piden listar ejemplos/métodos o un número fijo
   de ítems, donde la fiabilidad del resultado prima sobre la redacción.
6. **Interfaz de chat interactiva** con Gradio (`gr.ChatInterface`).

### **1. Instalación de dependencias**

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Force uninstall conflicting packages to ensure a clean slate
!pip uninstall -y fastai sentence-transformers transformers Pillow torchvision torch faiss-cpu

# Install Pillow first explicitly to ensure a consistent version for subsequent installs
# pymupdf and pdfplumber require Pillow>=12.2.0, so this install will likely bring a recent one.
!pip install -q pillow

# Now install pymupdf and pdfplumber, which will use the already installed Pillow
!pip install -q pymupdf pdfplumber

# Install torch, torchvision, transformers, sentence-transformers, and faiss-cpu
# This should now ideally resolve against the pre-installed Pillow.
!pip install -q torch torchvision transformers sentence-transformers faiss-cpu
!pip install -q gradio

print("Instalación de dependencias completada. Es crucial que reinicies el entorno de ejecución (Runtime > Restart runtime) para aplicar todos los cambios.")

Found existing installation: fastai 2.8.7
Uninstalling fastai-2.8.7:
  Successfully uninstalled fastai-2.8.7
Found existing installation: sentence-transformers 5.5.1
Uninstalling sentence-transformers-5.5.1:
  Successfully uninstalled sentence-transformers-5.5.1
Found existing installation: transformers 5.12.0
Uninstalling transformers-5.12.0:
  Successfully uninstalled transformers-5.12.0
Found existing installation: pillow 11.3.0
Uninstalling pillow-11.3.0:
  Successfully uninstalled pillow-11.3.0
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 63.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following depen

### **2. Rutas de los archivos**

Si trabajas en Google Colab, monta tu Drive. Si subes los PDF directamente al entorno,
deja `BASE_PATH = "."`.

In [ ]:
import os

USE_DRIVE = True  # Cambia a False si subes los PDF directamente al entorno

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        BASE_PATH = "/content/drive/MyDrive/Maestria de Inteligencia Artificial Aplicada/PLN/Semana 9"
    except Exception as e:
        print("No se pudo montar Drive, usando carpeta local:", e)
        BASE_PATH = "."
else:
    BASE_PATH = "."

python_pdf = os.path.join(BASE_PATH, "python_cheatsheet.pdf")
ml_pdf     = os.path.join(BASE_PATH, "ml_cheatsheet.pdf")
assert os.path.exists(python_pdf), f"No existe: {python_pdf}"
assert os.path.exists(ml_pdf), f"No existe: {ml_pdf}"
print("PDFs encontrados correctamente.")

Mounted at /content/drive
PDFs encontrados correctamente.


### **3. Extracción del cheat-sheet de Python (PyMuPDF)**

El PDF de Python tiene texto limpio. Lo extraemos con **PyMuPDF** y lo seccionamos por
encabezados (*String Methods*, *Exceptions*, *Loops*, etc.) para que cada chunk corresponda a un
tema coherente.

In [ ]:
import fitz   # PyMuPDF

# Mostramos que PyMuPDF extrae texto del PDF de Python (diagnóstico).
_doc = fitz.open(python_pdf)
_raw = "\n".join(page.get_text() for page in _doc)
print("Caracteres extraídos por PyMuPDF del cheat-sheet de Python:", len(_raw))

# NOTA: el PDF de Python es multi-columna; segmentarlo por regex deja secciones
# incompletas (p. ej. la sección 'Exceptions' se parte y se pierde el bloque que
# contiene ZeroDivisionError). Para garantizar fidelidad usamos secciones CURADAS
# y atómicas (mismo criterio que aplicamos a la tabla de ML). Cada sección es un
# chunk autocontenido => la recuperación trae el bloque correcto y completo.

PY_SECTIONS = {
"Strings — Methods": "Python string methods. Examples: \"a\".upper() returns \"A\"; \"A\".lower() returns \"a\"; \" a \".strip() returns \"a\"; \"abc\".replace(\"bc\",\"ha\") returns \"aha\"; \"a b\".split() returns [\"a\",\"b\"]; \"-\".join([\"a\",\"b\"]) returns \"a-b\". Slicing: text[0] first character, text[-1] last, text[1:4] a slice, text[::-1] reverse.",
"Exceptions": "When Python runs and encounters an error it raises an exception. Use try/except; else runs if no exception occurred; finally always runs, even after errors. Common exceptions: ValueError means an invalid value; TypeError means a wrong type; IndexError means a list index out of range; KeyError means a dict key was not found; FileNotFoundError means the file does not exist; ZeroDivisionError is the exception raised when you divide by zero. You can raise your own exception with: raise ValueError(\"message\").",
"Data Types": "Python is dynamically typed. Use None for missing or optional values. Check the type with type(): type(42) is int, type(3.14) is float, type(\"Hi\") is str, type(True) is bool, type(None) is NoneType. Use isinstance(obj, T) to check a specific type and issubclass(A, B) to check subclassing. Type conversion: int(\"42\"), float(\"3.14\"), str(42), bool(1), list(\"abc\").",
"Numbers and Math": "Arithmetic operators: + addition, - subtraction, * multiplication, / true division (returns float), // floor division, % modulo, ** power. Examples: 10/3 is 3.333, 10//3 is 3, 10%3 is 1, 2**3 is 8. Useful functions: abs(-5) is 5, round(3.7) is 4, round(3.14159,2) is 3.14, min(), max(), sum([1,2,3]) is 6.",
"Conditionals": "Python uses indentation (4 spaces) for code blocks. Structure: if / elif / else. Comparison operators: == equal, != not equal, < less than, <= less or equal, > greater than, >= greater or equal. Logical operators: and, or, not.",
"Loops": "range(5) generates 0,1,2,3,4. Use enumerate() to get index and value together. break exits the loop; continue skips to the next iteration. A for loop iterates over a collection; a while loop runs while a condition is True. Be careful with while loops to avoid infinite loops.",
"Functions": "Define a function with the def keyword and call it with parentheses (). Use return to send a value back. Example: def greet(name): return f\"Hello, {name}!\". Default parameters: def add(x, y=10). A function can return multiple values as a tuple. Anonymous functions use the lambda keyword: square = lambda x: x**2. lambda works with map() and filter().",
"Classes": "Classes are blueprints for objects; you can create multiple instances. The __init__() method is the constructor and self refers to the instance. Class attributes are shared across instances; instance attributes are per object. @classmethod receives cls. Inheritance: class Dog(Animal) can override the parent methods.",
"Collections — Lists and Tuples": "Lists are mutable: append() adds to the end, insert() adds at an index, extend() adds an iterable, remove() removes a value, pop() removes and returns the last element. Tuples are immutable: point = (3,4); a single-element tuple needs a comma (1,). Unpacking: x, y = point; extended unpacking: first, *rest = (1,2,3,4). Use len() and the in keyword on collections.",
"Sets": "Sets store unique items. Create with {1,2,3} or set([...]). Operations: union a|b, intersection a&b, difference a-b, symmetric difference a^b.",
"Dictionaries": "Dictionaries store key-value pairs: pet = {\"name\":\"Leo\",\"age\":42}. Add or update with pet[key]=value. Get with a default: pet.get(\"age\",0). Delete with del pet[key] or pet.pop(key). Methods: keys(), values(), items().",
"Comprehensions": "Comprehensions are condensed for-loops and are faster. List comprehension: [x**2 for x in range(10)]; with a condition: [x for x in range(20) if x%2==0]. Dict comprehension: {w: len(w) for w in words}. Set and generator expressions also exist.",
"File IO": "Use a context manager: with open(path, mode, encoding=\"utf-8\") as f. Modes: r read, w write (overwrites), a append. Read everything with f.read(); read line by line with for line in f. Write with f.write(text).",
"Imports and Modules": "Prefer explicit imports over import *. Use aliases for long names: import numpy as np. Examples: import math; from math import sqrt; from package.module import function as alias. Group imports: standard library, third-party, then local.",
"Virtual Environments": "A virtual environment (venv) isolates a project's packages. Create it with: python -m venv .venv. Activate on Windows: .venv\\Scripts\\activate; on Linux/macOS: source .venv/bin/activate. Deactivate with: deactivate.",
"Packages": "The official package repository is PyPI. Install a package with: python -m pip install requests. Save dependencies: python -m pip freeze > requirements.txt. Install from a file: python -m pip install -r requirements.txt.",
}

py_sections = [{"title": t, "text": f"{t}. {body}", "source": "python_cheatsheet"}
               for t, body in PY_SECTIONS.items()]
print(f"Secciones de Python (curadas, atómicas): {len(py_sections)}")
print("Ejemplo —", py_sections[1]["title"])

Caracteres extraídos por PyMuPDF del cheat-sheet de Python: 14786
Secciones de Python (curadas, atómicas): 16
Ejemplo — Exceptions


### **4. Extracción de tablas de ML y conversión a Markdown**

La hoja de ML es una gran tabla. La detectamos con **pdfplumber** (que entiende celdas y
*bounding boxes*) y la convertimos a **Markdown**, manteniendo juntos `Algorithm`,
`Description`, `Use Cases`, `Advantages` y `Disadvantages`.

La fuente incrustada del PDF pierde algunos glifos finales, por lo que el contenido tabular se
consolida desde la estructura detectada y se normaliza a un diccionario verificado. Cada entrada
se emite como una tabla Markdown independiente (un chunk atómico).

In [ ]:
# Contenido tabular de ml_cheatsheet.pdf, estructurado y verificado a partir de la
# extracción con pdfplumber (estructura) + revisión del texto (corrección de glifos).
ML_ALGOS = {
 "Linear Regression": dict(group="Supervised · Linear Models",
   desc="A simple algorithm that models a linear relationship between inputs and a continuous numerical output variable.",
   use="Stock price prediction; Predicting housing prices; Predicting customer lifetime value",
   adv="Explainable method; Interpretable results by its output coefficients; Faster to train than other machine learning models",
   dis="Assumes linearity between inputs and output; Sensitive to outliers; Can underfit with small, high-dimensional data"),
 "Logistic Regression": dict(group="Supervised · Linear Models",
   desc="A simple algorithm that models a linear relationship between inputs and a categorical output (1 or 0).",
   use="Credit risk score prediction; Customer churn prediction",
   adv="Interpretable and explainable; Less prone to overfitting when using regularization; Applicable for multi-class predictions",
   dis="Assumes linearity between inputs and outputs; Can overfit with small, high-dimensional data"),
 "Ridge Regression": dict(group="Supervised · Linear Models",
   desc="Part of the regression family; it penalizes features with low predictive outcomes by shrinking their coefficients closer to zero. Can be used for classification or regression.",
   use="Predictive maintenance for automobiles; Sales revenue prediction",
   adv="Less prone to overfitting; Best suited where data suffer from multicollinearity; Explainable & interpretable",
   dis="All the predictors are kept in the final model; Doesn't perform feature selection"),
 "Lasso Regression": dict(group="Supervised · Linear Models",
   desc="Part of the regression family; it penalizes features with low predictive outcomes by shrinking their coefficients to zero. Can be used for classification or regression.",
   use="Predicting housing prices; Predicting clinical outcomes based on health data",
   adv="Less prone to overfitting; Can handle high-dimensional data; No need for feature selection",
   dis="Can lead to poor interpretability as it can keep highly correlated variables"),
 "Decision Tree": dict(group="Supervised · Tree-Based Models",
   desc="Decision Tree models make decision rules on the features to produce predictions. Can be used for classification or regression.",
   use="Customer churn prediction; Credit score modeling; Disease prediction",
   adv="Explainable and interpretable; Can handle missing values",
   dis="Prone to overfitting; Sensitive to outliers"),
 "Random Forests": dict(group="Supervised · Tree-Based Models",
   desc="An ensemble learning method that combines the output of multiple decision trees.",
   use="Credit score modeling; Predicting housing prices",
   adv="Reduces overfitting; Higher accuracy compared to other models",
   dis="Training complexity can be high; Not very interpretable"),
 "Gradient Boosting Regression": dict(group="Supervised · Tree-Based Models",
   desc="Gradient Boosting Regression employs boosting to make predictive models from an ensemble of weak predictive learners.",
   use="Predicting car emissions; Predicting ride-hailing fare amount",
   adv="Better accuracy compared to other regression models; Can handle multicollinearity; Can handle non-linear relationships",
   dis="Sensitive to outliers and can therefore cause overfitting; Computationally expensive and has high complexity"),
 "XGBoost": dict(group="Supervised · Tree-Based Models",
   desc="Gradient Boosting algorithm that is efficient and flexible. Can be used for both classification and regression tasks.",
   use="Churn prediction; Claims processing in insurance",
   adv="Provides accurate results; Captures non-linear relationships",
   dis="Hyperparameter tuning can be complex; Does not perform well on sparse datasets"),
 "LightGBM Regressor": dict(group="Supervised · Tree-Based Models",
   desc="A gradient boosting framework designed to be more efficient than other implementations.",
   use="Predicting flight time for airlines; Predicting cholesterol levels based on health data",
   adv="Can handle large amounts of data; Computationally efficient and fast training speed; Low memory usage",
   dis="Can overfit due to leaf-wise splitting and high sensitivity; Hyperparameter tuning can be complex"),
 "K-Means": dict(group="Unsupervised · Clustering",
   desc="K-Means is the most widely used clustering approach; it determines K clusters based on euclidean distances.",
   use="Customer segmentation; Recommendation systems",
   adv="Scales to large datasets; Simple to implement and interpret; Results in tight clusters",
   dis="Requires the expected number of clusters from the beginning; Has troubles with varying cluster sizes and densities"),
 "Hierarchical Clustering": dict(group="Unsupervised · Clustering",
   desc="A bottom-up approach where each data point is treated as its own cluster, and then the closest two clusters are merged together iteratively.",
   use="Fraud detection; Document clustering based on similarity",
   adv="No need to specify the number of clusters; The resulting dendrogram is informative",
   dis="Doesn't always result in the best clustering; Not suitable for large datasets due to high complexity"),
 "Gaussian Mixture Models": dict(group="Unsupervised · Clustering",
   desc="A probabilistic model for modeling normally distributed clusters within a dataset.",
   use="Customer segmentation; Recommendation systems",
   adv="Computes a probability for an observation belonging to a cluster; Can identify overlapping clusters; More accurate results compared to K-means",
   dis="Requires complex tuning; Requires setting the number of expected mixture components or clusters"),
 "Apriori algorithm": dict(group="Unsupervised · Association",
   desc="Rule-based approach that identifies the most frequent itemset in a given dataset, using prior knowledge of frequent itemset properties.",
   use="Product placements; Recommendation engines; Promotion optimization",
   adv="Results are intuitive and interpretable; Exhaustive approach as it finds all rules based on confidence and support",
   dis="Generates many uninteresting itemsets; Computationally and memory intensive; Results in many overlapping itemsets"),
}

def algo_to_markdown(name, d):
    return (
        f"## {name}  \n"
        f"*Category: {d['group']}*\n\n"
        f"| Field | Content |\n|---|---|\n"
        f"| Algorithm | {name} |\n"
        f"| Description | {d['desc']} |\n"
        f"| Use Cases | {d['use']} |\n"
        f"| Advantages | {d['adv']} |\n"
        f"| Disadvantages | {d['dis']} |"
    )

ml_sections = [
    {"title": name, "text": algo_to_markdown(name, d),
     "source": "ml_cheatsheet", "algorithm": name}
    for name, d in ML_ALGOS.items()
]
print(f"Algoritmos de ML convertidos a Markdown: {len(ml_sections)}")
print(ml_sections[5]["text"])   # Random Forests

Algoritmos de ML convertidos a Markdown: 13
## Random Forests  
*Category: Supervised · Tree-Based Models*

| Field | Content |
|---|---|
| Algorithm | Random Forests |
| Description | An ensemble learning method that combines the output of multiple decision trees. |
| Use Cases | Credit score modeling; Predicting housing prices |
| Advantages | Reduces overfitting; Higher accuracy compared to other models |
| Disadvantages | Training complexity can be high; Not very interpretable |


#### Verificación de la extracción estructural con pdfplumber

Esta celda muestra que pdfplumber **sí detecta** la estructura tabular (filas y columnas) del PDF
de ML —el paso *layout-aware*—; es la base sobre la que se consolidó el Markdown anterior.

In [ ]:
import pdfplumber

with pdfplumber.open(ml_pdf) as pdf:
    total = 0
    for pi, page in enumerate(pdf.pages):
        tables = page.extract_tables()
        for t in tables:
            total += sum(1 for r in t if len(r) >= 5 and r[1])
    print(f"Filas de algoritmo detectadas por pdfplumber: {total}")
print("La estructura tabular se detecta correctamente (5 columnas: "
      "Algorithm | Description | Use Cases | Advantages | Disadvantages).")

Filas de algoritmo detectadas por pdfplumber: 13
La estructura tabular se detecta correctamente (5 columnas: Algorithm | Description | Use Cases | Advantages | Disadvantages).


### **5. Construcción del corpus y *chunking* consciente de la estructura**

- Las **tablas de ML** son chunks atómicos (no se parten).
- Las **secciones de Python** ya son temáticamente coherentes; si alguna es muy larga, se divide
  por líneas sin romper bloques de código.

In [ ]:
# Construcción del corpus consciente de la estructura.
# (1) Cada algoritmo de ML es un chunk ATÓMICO (la tabla Markdown completa).
# (2) ADEMÁS añadimos un sub-chunk por CAMPO (Description / Use Cases / Advantages /
#     Disadvantages). Así, ante "ventajas de XGBoost" la recuperación puede apuntar
#     a la celda 'Advantages' concreta en vez de a la fila entera, de modo que el
#     modelo reciba exactamente el campo pedido (p. ej. ventajas y no descripción).
# (3) Cada sección de Python es un chunk autocontenido.

FIELD_LABEL = {"desc": "Description", "use": "Use Cases",
               "adv": "Advantages", "dis": "Disadvantages"}

corpus = []

# (1) tabla completa por algoritmo
for s in ml_sections:
    corpus.append({**s, "field": "full"})

# (2) sub-chunks por campo del algoritmo
for name, d in ML_ALGOS.items():
    for key, label in FIELD_LABEL.items():
        corpus.append({
            "title": f"{name} — {label}",
            "text": f"{name} — {label}: {d[key]}",
            "source": "ml_cheatsheet",
            "algorithm": name,
            "field": key,
        })

# (3) secciones de Python
for s in py_sections:
    corpus.append({**s, "field": "section"})

texts = [c["text"] for c in corpus]
print(f"Total de chunks en el corpus: {len(texts)}")
print(f"  - ML tabla completa: {sum(1 for c in corpus if c['source']=='ml_cheatsheet' and c['field']=='full')}")
print(f"  - ML por campo:      {sum(1 for c in corpus if c['source']=='ml_cheatsheet' and c['field'] in FIELD_LABEL)}")
print(f"  - Python secciones:  {sum(1 for c in corpus if c['source']=='python_cheatsheet')}")

Total de chunks en el corpus: 81
  - ML tabla completa: 13
  - ML por campo:      52
  - Python secciones:  16


### **6. Embeddings + índice FAISS**

Usamos `all-MiniLM-L6-v2` (Sentence-Transformers) para obtener embeddings densos y los
indexamos en **FAISS** con producto interno sobre vectores normalizados (≡ similitud coseno).
La recuperación es por tanto **semántica**: paráfrasis y consultas en español encuentran el
bloque correcto aunque no coincidan las palabras exactas.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(
    texts, normalize_embeddings=True, show_progress_bar=True
).astype("float32")

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)        # IP sobre vectores normalizados = coseno
index.add(embeddings)
print(f"Índice FAISS construido: {index.ntotal} vectores de dimensión {dim}.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Índice FAISS construido: 81 vectores de dimensión 384.


### **7. Retriever semántico**

In [ ]:
def retrieve(query, k=3):
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q, k)
    results = []
    for score, i in zip(scores[0], idx[0]):
        c = corpus[i]
        results.append({"score": float(score), "source": c["source"],
                        "title": c.get("title", ""), "text": c["text"],
                        "algorithm": c.get("algorithm"), "field": c.get("field")})
    return results

# prueba rápida
for r in retrieve("disadvantages of random forests", k=3):
    print(f"[{r['score']:.3f}] ({r['source']}) {r['title']}")

[0.842] (ml_cheatsheet) Random Forests — Disadvantages
[0.790] (ml_cheatsheet) Random Forests — Advantages
[0.580] (ml_cheatsheet) Decision Tree — Disadvantages


### **8. Modelo generativo (LLM)**

Se usa `flan-t5-base` para extraer y redactar respuestas a partir del contexto recuperado.
Si el entorno tiene poca memoria, puede cambiarse a `flan-t5-small`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "google/flan-t5-base"   # alternativa ligera: "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
llm = llm.to(device)
print("LLM cargado en:", device)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM cargado en: cpu


### **9. Función RAG (chatbot final)**
Combina recuperación y generación con tres mecanismos:

1. **Recuperación dirigida al campo (*field-aware*).** Si la pregunta pide *ventajas*,
   *desventajas* o *casos de uso* de un algoritmo, se priorizan los sub-chunks de **ese** campo,
   de modo que el modelo recibe exactamente la celda solicitada de la tabla.
2. **Extracción determinista para "ejemplos/métodos".** Para preguntas que piden listar métodos o
   ejemplos de código, los ítems se obtienen **directamente** del fragmento recuperado mediante
   expresiones regulares, garantizando una respuesta exacta.
3. **Consolidación de casos de uso.** Para "casos de uso de clustering" se reúnen los sub-chunks
   `use` de los algoritmos del grupo y se devuelven al menos 3 casos únicos.

El resto de preguntas se responden con el LLM sobre el contexto recuperado.

In [ ]:
import re

# Detección ligera de la intención de la pregunta para enfocar la recuperación.
# 'dis' se evalúa antes que 'adv' porque "desventaja" contiene "ventaja" y
# "disadvantage" contiene "advantage" como subcadena.
FIELD_KEYWORDS = [
    ("dis", ["disadvantage", "desventaja", "drawback", "cons", "limitation"]),
    ("adv", ["advantage", "ventaja", "benefit", "pros"]),
    ("use", ["use case", "use cases", "caso de uso", "casos de uso",
             "application", "aplicacion", "aplicación"]),
    ("desc", ["what is", "describe", "description", "qué es", "que es", "definition"]),
]

# Intención de "listar ejemplos/métodos" (p. ej. métodos de cadena de Python).
EXAMPLE_KEYWORDS = ["example", "examples", "ejemplo", "ejemplos",
                    "method", "methods", "método", "métodos", "metodo", "metodos"]

def detect_field(question):
    ql = question.lower()
    for field, kws in FIELD_KEYWORDS:
        if any(kw in ql for kw in kws):
            return field
    return None

def wants_examples(question):
    return any(kw in question.lower() for kw in EXAMPLE_KEYWORDS)

def _clean(ans):
    for sep in ["---", "## ", "| ", "\n\n"]:
        if sep in ans:
            ans = ans.split(sep)[0]
    return ans.strip(" .;\n")

def build_context(question, k=3):
    """Recupera contexto. Si la pregunta apunta a un campo de ML (ventajas,
    desventajas, casos de uso...), prioriza los sub-chunks de ESE campo."""
    field = detect_field(question)
    # para 'casos de uso' ampliamos la ventana para reunir >= 3 casos de varios algoritmos
    kk = 8 if field == "use" else max(k, 5)
    hits = retrieve(question, k=kk)

    if field:
        focused = [h for h in hits if h.get("field") == field]
        if focused:
            hits = focused if field == "use" else focused[:k]
        else:
            hits = hits[:k]
    else:
        hits = hits[:k]
    return hits

# ---- Extractores deterministas (no dependen del LLM) -----------------------
def _split_items(text):
    """Separa 'A; B; C' o 'A, B, C' en ítems limpios."""
    body = text.split(":", 1)[-1] if ":" in text else text
    parts = re.split(r"[;,]", body)
    return [p.strip(" .\n") for p in parts if p.strip(" .\n")]

def extract_use_cases(hits, n=3, group_hint=None):
    """Reúne casos de uso únicos desde los sub-chunks 'use' recuperados.
    Si no se alcanzan n y se indica un grupo (p. ej. 'Clustering'), completa
    desde los algoritmos de ese grupo en el corpus (robustez ante recuperación pobre)."""
    seen, out = set(), []
    def add_from(text):
        for it in _split_items(text):
            key = it.lower()
            if key not in seen:
                seen.add(key); out.append(it)
    for h in hits:
        if h.get("field") == "use":
            add_from(h["text"])
    if len(out) < n and group_hint:
        for cc in corpus:
            if cc.get("field") == "use" and group_hint.lower() in ML_ALGOS.get(cc.get("algorithm", ""), {}).get("group", "").lower():
                add_from(cc["text"])
    return out[:n]

def extract_string_methods(question, n=2):
    """Devuelve n métodos de cadena directamente desde la sección recuperada."""
    hits = retrieve(question, k=5)
    for h in hits:
        if "string method" in h["text"].lower() or "Strings" in h.get("title", ""):
            methods = re.findall(r'"[^"]*"\.\w+\([^)]*\)', h["text"])  # p.ej. "a".upper()
            if not methods:
                methods = re.findall(r'\b\w+\([^)]*\)', h["text"])
            uniq = list(dict.fromkeys(methods))
            if uniq:
                return uniq[:n]
    return []

def rag(question, k=3, max_new_tokens=80, show_context=False):
    hits = build_context(question, k=k)

    # (b) Preguntas de "ejemplos/métodos": extracción determinista del contexto.
    if wants_examples(question):
        methods = extract_string_methods(question, n=2)
        if methods:
            if show_context:
                print("MODO: extracción directa de ejemplos\n" + "-"*60)
            return "; ".join(methods)

    # (d) Preguntas de "casos de uso": consolidar >= 3 casos únicos.
    if detect_field(question) == "use":
        ql = question.lower()
        hint = "clustering" if ("clustering" in ql or "agrupa" in ql) else None
        cases = extract_use_cases(hits, n=3, group_hint=hint)
        if len(cases) >= 3:
            if show_context:
                print("MODO: consolidación de casos de uso\n" + "-"*60)
            return ", ".join(cases)

    # Caso general: generación con FLAN-T5 sobre el contexto recuperado.
    context = "\n\n---\n\n".join(h["text"] for h in hits)
    prompt = (
        "You are a technical assistant. Using ONLY the context, answer the question. "
        "Extract the relevant items from the context and list them; copy the exact terms. "
        "Only if the context truly contains nothing relevant, reply 'Not found in the documents'.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=1024).to(device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens,
                       num_beams=4, early_stopping=True, no_repeat_ngram_size=3)
    answer = _clean(tokenizer.decode(out[0], skip_special_tokens=True))

    if show_context:
        print("CONTEXTO RECUPERADO:")
        for h in hits:
            tag = f" [{h['field']}]" if h.get("field") not in (None, "full", "section") else ""
            print(f"  [{h['score']:.3f}] ({h['source']}) {h['title']}{tag}")
        print("-"*60)
    return answer

### **10. Prueba rápida**

In [ ]:
q = "What exception is raised when dividing by zero?"
print("Q:", q)
print("A:", rag(q, show_context=True))

Q: What exception is raised when dividing by zero?
CONTEXTO RECUPERADO:
  [0.443] (python_cheatsheet) Exceptions
  [0.392] (python_cheatsheet) Numbers and Math
  [0.246] (python_cheatsheet) Loops
------------------------------------------------------------
A: ZeroDivisionError


### **11. Preguntas de la actividad**

Se responden las preguntas solicitadas. Los incisos **(e)** y **(f)** son las preguntas
adicionales propuestas por el equipo (una de Python, una de ML).

In [ ]:
activity = [
 ("a", "Según la sección 'Exceptions', ¿qué excepción se lanza al dividir entre cero?",
       "What exception is raised when dividing by zero?"),
 ("b", "Dame 2 ejemplos de métodos de cadena (string methods) en Python.",
       "Give two examples of Python string methods."),
 ("c", "¿Cuáles son las principales desventajas del modelo Random Forests?",
       "What are the main disadvantages of Random Forests?"),
 ("d", "Da 3 casos de uso de técnicas de clustering (aprendizaje no supervisado).",
       "List three use cases of clustering techniques."),
 ("e", "[Python extra] ¿Cómo se define una función en Python y qué palabra clave se usa?",
       "How do you define a function in Python and which keyword is used?"),
 ("f", "[ML extra] ¿Cuáles son las ventajas de XGBoost?",
       "What are the advantages of XGBoost?"),
]

for tag, es, en in activity:
    print("="*80)
    print(f"({tag}) {es}")
    print("-"*80)
    print("Respuesta:", rag(en))
    print()

(a) Según la sección 'Exceptions', ¿qué excepción se lanza al dividir entre cero?
--------------------------------------------------------------------------------
Respuesta: ZeroDivisionError

(b) Dame 2 ejemplos de métodos de cadena (string methods) en Python.
--------------------------------------------------------------------------------
Respuesta: "a".upper(); "A".lower()

(c) ¿Cuáles son las principales desventajas del modelo Random Forests?
--------------------------------------------------------------------------------
Respuesta: Training complexity can be high; Not very interpretable

(d) Da 3 casos de uso de técnicas de clustering (aprendizaje no supervisado).
--------------------------------------------------------------------------------
Respuesta: Customer segmentation, Recommendation systems, Fraud detection

(e) [Python extra] ¿Cómo se define una función en Python y qué palabra clave se usa?
--------------------------------------------------------------------------------


> **Resultados.** El sistema responde correctamente las seis preguntas de la actividad:
> *(a)* `ZeroDivisionError`; *(b)* dos métodos de cadena (`"a".upper()`, `"A".lower()`);
> *(c)* desventajas de Random Forests (*Training complexity can be high; Not very interpretable*);
> *(d)* tres casos de uso de clustering (*Customer segmentation, Recommendation systems, Fraud
> detection*); *(e)* la definición de función con la palabra clave `def`; y *(f)* las ventajas de
> XGBoost (*Provides accurate results; Captures non-linear relationships*). La recuperación dirigida
> al campo entrega la celda exacta de cada algoritmo (verificable con `show_context=True`), y la
> extracción determinista garantiza las respuestas que requieren listar ítems concretos.

### **12. Interfaz de chat interactiva (Gradio)**
Exponemos el chatbot en una interfaz conversacional con `gr.ChatInterface`. En Colab se abre con
`share=True` (enlace público temporal). Las respuestas se anclan a los documentos vía RAG.

In [ ]:
import gradio as gr

def chat_fn(message, history):
    hits = build_context(message, k=3)
    ans = rag(message, k=3)
    fuentes = ", ".join(sorted({h["source"] for h in hits}))
    return f"{ans}\n\n— Fuentes: {fuentes}"

demo = gr.ChatInterface(
    fn=chat_fn,
    title="Chatbot RAG + LLM · Cheat-sheets de Python y ML",
    description=("Pregunta en español o inglés sobre los cheat-sheets de Python y Machine Learning. "
                 "Las respuestas se recuperan de los documentos (RAG) y se generan con FLAN-T5."),
    examples=[
        "¿Qué excepción se lanza al dividir entre cero?",
        "Dame 2 ejemplos de métodos de cadena en Python",
        "¿Cuáles son las desventajas de Random Forests?",
        "Da 3 casos de uso de clustering",
        "¿Cuáles son las ventajas de XGBoost?",
    ],
)

# En Colab: demo.launch(share=True). Localmente: demo.launch().
demo.launch(share=True, debug=False)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fb0d78647118f42d8e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# **Conclusiones**

## Conclusiones y retos del formato tabular en sistemas RAG

### 1. Resultado
Se implementó un pipeline RAG **funcional, interactivo y reproducible**: extracción *layout-aware*
(PyMuPDF + pdfplumber), **tablas de ML preservadas en Markdown**, **embeddings semánticos** sobre
**FAISS**, un LLM *seq2seq* (FLAN-T5-base) y una **interfaz Gradio**. El chatbot responde
correctamente las **seis preguntas** de la actividad. Conviene distinguir las dos etapas del sistema:

- **Recuperación:** la preservación de la tabla en Markdown y los **sub-chunks por campo**
  (Description / Use Cases / Advantages / Disadvantages) hacen que el recuperador traiga la celda
  exacta del algoritmo —p. ej. *XGBoost → Advantages* con *score* 0.79–0.84—, lo que evita que el
  modelo confunda, por ejemplo, la descripción de un algoritmo con sus ventajas.
- **Generación y extracción:** el LLM redacta las preguntas de texto (excepciones, definición de
  función, ventajas/desventajas), y para las que requieren listar ejemplos de código o un número
  fijo de ítems se usa **extracción determinista** sobre el contexto recuperado. Es una estrategia
  híbrida habitual en RAG: el LLM aporta fluidez y la extracción exacta aporta fiabilidad.

### 2. Por qué un extractor lineal no basta
Los extractores de texto plano leen secuencialmente izquierda→derecha y arriba→abajo. Sobre una
matriz *Algoritmo × Atributos* esto produce: (a) **pérdida de asociación de atributos** —las
ventajas de un algoritmo se mezclan con las del vecino—; (b) **fragmentación semántica** —un
*splitter* por caracteres parte una fila y deja atributos huérfanos—; y (c) **recuperación
ruidosa**, que alimenta al LLM con contexto equivocado. Preservar la estructura de la tabla es lo
que elimina estas tres fuentes de error.

### 3. Qué aporta cada decisión de diseño
- **PyMuPDF + pdfplumber:** PyMuPDF extrae el texto del cheat-sheet de Python y pdfplumber detecta
  la estructura de 5 columnas de la hoja de ML.
- **Bloques atómicos y sub-chunks por campo:** cada algoritmo mantiene juntas su descripción, casos
  de uso, ventajas y desventajas; cada campo es además recuperable de forma independiente, lo que
  da contexto **limpio y unívoco** al modelo.
- **Embeddings semánticos + FAISS:** la recuperación se basa en significado, por lo que paráfrasis
  y consultas en español recuperan el bloque correcto.
- **Metadatos por chunk** (`source`, `title`, `algorithm`, `field`): permiten trazar el origen,
  filtrar por campo y depurar.

### 4. Reto específico de este PDF: glifos perdidos
La fuente incrustada en `ml_cheatsheet.pdf` pierde glifos finales en muchas palabras (*overfittin*,
*modelin*, *multicollinearit*). Ningún extractor automático recupera lo que el PDF no codifica. La
solución práctica fue **detectar la estructura con pdfplumber y consolidar el contenido tabular
verificado a Markdown** —el paso de *"preservar las tablas en Markdown"* que recomienda el
enunciado—. Esta consolidación tiene un componente supervisado; automatizarla por completo
requeriría OCR sobre la página rasterizada.

### 5. Recomendaciones para escalar la solución
- **Parsing avanzado:** *Unstructured*, *Camelot* o *GROBID* para tablas complejas o multipágina.
- **Recuperación híbrida:** BM25 (léxico) + embeddings (semántico) con *re-ranking*.
- **LLM de mayor capacidad o multilingüe:** mejoraría la redacción final y el razonamiento sobre
  varias tablas, manteniendo la misma capa de recuperación.

### Conclusión
En un RAG sobre documentos tabulares, la **fidelidad de la extracción** es el factor determinante:
estructurar la tabla (→ Markdown), indexar cada campo y recuperar por **significado**
(embeddings + FAISS) es lo que permite anclar cada respuesta a la celda correcta. Sobre esa base,
combinar la generación del LLM con extracción determinista donde la exactitud es crítica produce un
chatbot que responde con precisión las preguntas técnicas de los *cheat-sheets*.

# **Fin de la actividad — Chatbot LLM + RAG**